In [1]:
import pandas as pd 
import os 
import json 
import numpy as np
from os.path import dirname

pd.set_option("display.max_columns", None)
root_path = dirname(os.getcwd())
data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/comuzzi/_processed"
data_dir_graphs = root_path + "/data/datasets/comuzzi/graphs_repair/"
data_dir_ablation = root_path + "/data/datasets/ablation/" 
os.makedirs(data_dir_ablation, exist_ok=True)
print(root_path, data_dir_processed, data_dir_graphs, data_dir_ablation, sep="\n")

/home/danbi/Projects/SANAGRAPH
/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/_processed
/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/graphs_repair/
/home/danbi/Projects/SANAGRAPH/data/datasets/ablation/


In [2]:
from pipeline import (
    build_split_graphs,
    build_test_graphs_for_type,
    get_feature_variants,
    convert_feature_name,
    build_one_hot_encoders,
    group_by_case
)

In [3]:
with open("dataset_features.json", 'r') as file:
    dataset_info = json.load(file)

In [4]:
list(dataset_info.keys())

['BPI_Challenge_2013_open_problems',
 'sp2020',
 'Helpdesk',
 'BPI20_RequestForPayment',
 'BPI Challenge 2017 - Offer log',
 'BPI_Challenge_2012_W_Complete',
 'BPI_Challenge_2012_A',
 'bpi_2012_CZ',
 'bpi_2013_CZ',
 'large_log_CZ',
 'small_log_CZ',
 'sp2020_CZ',
 'BPI20_RequestForPayment_CZ']

In [5]:
dataset = "small_log_CZ"

In [6]:
ACT_TIME_ONLY = False

In [7]:
dataset_info = dataset_info[dataset]

In [8]:
variants = get_feature_variants(dataset_info)
print(f"{len(variants)} Variants to generate for {dataset}:")
for excluded_feature, cat, num in variants:
    print(f"  - ablate_{convert_feature_name(excluded_feature)} (excluded feature: {excluded_feature})")

2 Variants to generate for small_log_CZ:
  - ablate_Activity (excluded feature: Activity)
  - ablate_time_timestamp (excluded feature: time:timestamp)


In [9]:
nan_methods = ["odd", "even", "random", "window", "attr_level"]
masked_datasets = {key: pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_masked_{key}_all.csv") for key in nan_methods}

tab_all = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_all.csv")
tab_train = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_train.csv")
tab_valid = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_valid.csv")
tab_test = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_test.csv")

In [10]:
all_categorical = dataset_info["categorical"]

for k in all_categorical:
    tab_all[k] = tab_all[k].astype("object")
    tab_train[k] = tab_train[k].astype("object")
    tab_valid[k] = tab_valid[k].astype("object")
    tab_test[k] = tab_test[k].astype("object")
    for k_m in masked_datasets:
        masked_datasets[k_m][k] = masked_datasets[k_m][k].astype("object")

tab_train["CaseID"] = tab_train["CaseID"].astype(np.str_)
tab_valid["CaseID"] = tab_valid["CaseID"].astype(np.str_)
tab_test["CaseID"] = tab_test["CaseID"].astype(np.str_)
for k_m in masked_datasets:
    masked_datasets[k_m]["CaseID"] = masked_datasets[k_m]["CaseID"].astype(np.str_)

In [12]:
encoders = build_one_hot_encoders(tab_all, all_categorical)
grouped_train = group_by_case(tab_train)
grouped_valid = group_by_case(tab_valid)
grouped_test = group_by_case(tab_test)
grouped_masked = {method: group_by_case(df) for method, df in masked_datasets.items()}

In [11]:
import pickle

In [14]:
import shutil
shutil.rmtree(data_dir_ablation + "small_log_CZ/", ignore_errors=True)

In [15]:
expected_files = [f"{split}_V2_repair.pkl" for split in ["TRAIN", "VALID", "TEST"]] + \
                  [f"TEST_V2_repair_{t}.pkl" for t in nan_methods]

for excluded_feature, categorical_columns, real_value_columns in variants:

    safe_feature = convert_feature_name(excluded_feature)
    variant_dir = data_dir_ablation + f"{dataset}/ablate_{safe_feature}/"

    already_generated = os.path.isdir(variant_dir) and all(
        os.path.exists(variant_dir + fname) for fname in expected_files
    )

    if already_generated:
        print(f"Variant ablate_{safe_feature} already generated, skip.")
        continue

    os.makedirs(variant_dir, exist_ok=True)
    print(f"\n=== Variant: ablate_{safe_feature} ===")

    for split_name, grouped_split in [("TRAIN", grouped_train), ("VALID", grouped_valid), ("TEST", grouped_test)]:
        print(f"  {split_name}...")
        X = build_split_graphs(grouped_split, categorical_columns, real_value_columns, grouped_masked, nan_methods, encoders)
        with open(variant_dir + f"{split_name}_V2_repair.pkl", "wb") as f:
            pickle.dump(X, f)
        del X

    for test_type in nan_methods:
        print(f"  TEST ({test_type})...")
        X = build_test_graphs_for_type(grouped_test, categorical_columns, real_value_columns, grouped_masked, nan_methods, encoders, mask_type=test_type)
        with open(variant_dir + f"TEST_V2_repair_{test_type}.pkl", "wb") as f:
            pickle.dump(X, f)
        del X

print("\nGenerated all the variants.")


=== Variant: ablate_Activity ===
  TRAIN...


  0%|          | 0/1200 [00:00<?, ?it/s]

  VALID...


  0%|          | 0/400 [00:00<?, ?it/s]

  TEST...


...

  TEST (window)...


  0%|          | 0/400 [00:00<?, ?it/s]

  TEST (attr_level)...


  0%|          | 0/400 [00:00<?, ?it/s]


Generated all the variants.
